# XWhy Tabular Explainer Example

This notebook demonstrates how to use the `TabularExplainer` from the `xwhy` library.
We will train an XGBoost regression model on the Boston Housing dataset and explain a single prediction.

## Setup

Run the next cell first on Google Colab or Kaggle (internet access required). It installs XWhy from GitHub; you can skip it if XWhy is already installed locally.

In [ ]:
# Setup for Google Colab / Kaggle: install the latest XWhy from GitHub.
# Skip this cell if XWhy is already installed in your environment.
%pip install -q "xwhy @ git+https://github.com/Dependable-Intelligent-Systems-Lab/xwhy.git"

from importlib.metadata import version
print("xwhy", version("xwhy"))

In [ ]:
import xwhy
import numpy as np
from sklearn.datasets import fetch_openml
import xgboost
from xwhy import TabularExplainer

## 1. Data Loading and Preprocessing

We define a `StandardScalerWrapper` to ensure our inputs are scaled, which is recommended for the explainer's distance metrics.

In [ ]:
# Standard Scaler
class StandardScalerWrapper:
    """
    Simple reusable normalization utility for SMILE/LIME/SHAP consistency.
    """

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0) + 1e-8
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return (X - self.mean_) / self.std_

    def fit_transform(self, X):
        return self.fit(X).transform(X)

# Load Boston housing dataset
bunch = fetch_openml(name="boston", version=1, as_frame=True, parser="auto")
X = bunch.data
y = bunch.target.rename("MEDV")
scaler = StandardScalerWrapper()
X_scaled = scaler.fit_transform(X.values)

## 2. Model Training

In [ ]:
# Train an XGBoost Regressor
xg_model = xgboost.XGBRegressor(
    enable_categorical=True,
    tree_method="hist",
    random_state=42,
    n_estimators=100
).fit(X, y)

## 3. Explanation Generation

We'll explain the very first instance in our dataset.

In [ ]:
instance = X_scaled[0]

try:
    explainer = TabularExplainer(model=xg_model)

    result = explainer.explain(instance=instance, feature_names=X.columns.to_list())
    
    print(result.metrics)
    print("Explanation successful!")

except Exception as e:
    print(f"Error during pipeline execution: {e}")
    result = None

# Visualizations

In [ ]:
result.plot()

In [ ]:
xwhy.plots.plot_explanation_waterfall(result)

In [ ]:
xwhy.plots.plot_feature_contributions(result)

In [ ]:
xwhy.plots.plot_method_contributions(result)

In [ ]:
xwhy.plots.bar(result)

In [ ]:
xwhy.plots.waterfall(result)

In [ ]:
xwhy.plots.text(result)

In [ ]:
xwhy.plots.force(result)

In [ ]:
xwhy.plots.decision(result)

In [ ]:
xwhy.plots.plot_feature_bar_chart(result)

In [ ]:
xwhy.plots.plot_feature_box_plot(result)